In [2]:
import json
import subprocess
from pathlib import Path
from llm_core.config import RAW_DATA_DIR

2025-07-06 08:27:20.101 | INFO     | llm_core.config:<module>:11 - PROJ_ROOT path is: /home/arys/projects/fictional-univers-builder/llm_core
2025-07-06 08:27:20.102 | INFO     | llm_core.config:<module>:35 - MLFLOW URI path is: file:///home/arys/projects/fictional-univers-builder/llm_core/mlflow_root/mlruns


In [3]:
output_path = RAW_DATA_DIR / "dataset_fiction_local.jsonl"
model_name = "mistral"
num_batches = 100000
examples_per_batch = 10

In [4]:
base_prompt = f"""
Tu es un générateur de dataset pour fine-tuner un LLM assistant à la création d’univers de fiction.

Génère des exemples réalistes dans le format suivant :
---
[question] : une question posée par un créateur d’univers fictif.
[context] : un extrait de documentation existante (ou "Aucun" si le LLM devra inventer).
[answer] : une réponse complète et cohérente :
  - si le contexte contient l'information, le LLM doit s'y référer strictement ;
  - sinon, le LLM invente un élément original et logique.

Donne {examples_per_batch} exemples couvrant des thèmes comme géographie, politique, magie, races, culture, etc.
Utilise un style clair et immersif.
"""


In [5]:
def query_ollama(prompt, model="llama3"):
    result = subprocess.run(
        ["ollama", "run", model, prompt],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    return result.stdout.strip()


In [6]:
def parse_examples(text):
    examples = []
    current = {}
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("[question]"):
            current["question"] = line[len("[question] :"):].strip()
        elif line.startswith("[context]"):
            current["context"] = line[len("[context] :"):].strip()
        elif line.startswith("[answer]"):
            current["answer"] = line[len("[answer] :"):].strip()
        elif line == "" and current:
            if all(k in current for k in ("question", "context", "answer")):
                examples.append(current)
            current = {}
    if all(k in current for k in ("question", "context", "answer")):
        examples.append(current)
    return examples

In [7]:
from tqdm import tqdm
i = 0
with open(output_path, "a", encoding="utf-8") as f:

    for i in tqdm(range(num_batches), desc=f"🧠 Batch : {num_batches} avec {model_name}"):

        result = query_ollama(base_prompt, model=model_name)
        examples = parse_examples(result)

        for ex in examples:
            json.dump(ex, f, ensure_ascii=False)
            f.write("\n")

🧠 Batch : 100000 avec mistral:   3%|▎         | 2662/100000 [10:06:05<369:22:03, 13.66s/it]


KeyboardInterrupt: 